# BT4012 Kaggle 2026 — Illicit Bitcoin Transaction Detection (final pipeline)

**Student:** e1139091 · **Metric:** ROC AUC · **Runtime:** about 15 minutes on a 12-thread laptop (`RUN_VALIDATION = False` skips Section 5, about 5 minutes).

This notebook regenerates **both submissions selected for the final private leaderboard**:

| file | model | public AUC |
| --- | --- | --- |
| `exp11c_seedavg5.csv` | two-stage LightGBM, model B with `extra_trees`, 600 trees, learning rate 0.025, 5 seeds | 0.96925 |
| `exp11g_rankavg_11abc.csv` | equal-weight rank average of exp11a, exp11b, exp11c (three 5-seed variants) | 0.96873 |

Pipeline: (1) load data → (2) structural graph features → (3) visibility-masked neighbour aggregates → (4) two-stage model with out-of-fold neighbour risk scores → (5) forward-in-time validation of the final models → (6) fit on all labelled steps and write the CSVs → (7) check against the submitted files.

The full experiment history behind these choices (30+ experiments, Sections 1–17) is in `experiments.ipynb`; results are in `experiment_log.csv`.

*AI use: Claude (Claude Code) wrote most of this code to my specification; design and modelling decisions were mine (see report).*

In [1]:
import time
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

SEED = 42
SEEDS_FINAL = [42, 43, 44, 45, 46]        # final models: probabilities averaged over 5 seeds
SEEDS_CV = [42, 43, 44]                   # validation in Section 5 (as reported)
RUN_VALIDATION = True                     # False = skip Section 5 and only regenerate the submissions

DATA = Path('data')                       # train.csv, test.csv, txs_edgelist.csv, sample_submission.csv
OUT = Path('submissions') / 'final'
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time()

## 1. Load data

- `label` is blank for unknown transactions (78% of training rows). Unknown is **not** licit: those rows are never used as training targets or in evaluation.
- No feature has missing values and features are already normalised, so no imputation or scaling.
- `time_step` is used only to split data, never as a feature (test steps 36–49 never occur in training).

In [2]:
feat_cols = [f'feat_{i}' for i in range(1, 166)]
dtypes = {c: 'float32' for c in feat_cols}

train = pd.read_csv(DATA / 'train.csv', dtype=dtypes)
test = pd.read_csv(DATA / 'test.csv', dtype=dtypes)
edges = pd.read_csv(DATA / 'txs_edgelist.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')

lab = train[train['label'].notna()].copy()
lab['label'] = lab['label'].astype(int)
assert (test['index'].values == sample['index'].values).all(), 'test order differs from sample submission'
print('train', train.shape, '| labelled', lab.shape, f"({lab.label.mean():.1%} illicit) | test", test.shape, '| edges', edges.shape)

train (141772, 168) | labelled (31235, 168) (11.7% illicit) | test (15329, 168) | edges (234355, 2)


## 2. Structural graph features

An edge `txId1 → txId2` means an output of `txId1` is spent by `txId2`. Every edge stays within one time step and the graph is acyclic.
These 12 features use only the edge list, which covers every transaction in both periods, so they are built identically for train and test.

In [3]:
G = nx.DiGraph()
G.add_edges_from(edges.itertuples(index=False, name=None))
step = pd.concat([train[['txId', 'time_step']], test[['txId', 'time_step']]]).set_index('txId')['time_step']

gf = pd.DataFrame(index=pd.Index(step.index, name='txId'))
in_deg = edges.txId2.value_counts()
out_deg = edges.txId1.value_counts()
gf['in_deg'] = in_deg.reindex(gf.index).fillna(0)
gf['out_deg'] = out_deg.reindex(gf.index).fillna(0)
gf['deg_ratio'] = (gf.out_deg + 1) / (gf.in_deg + 1)
gf['is_source'] = (gf.in_deg == 0).astype(int)
gf['is_sink'] = (gf.out_deg == 0).astype(int)
gf['in2_paths'] = edges.assign(d=edges.txId1.map(in_deg).fillna(0)).groupby('txId2').d.sum().reindex(gf.index).fillna(0)
gf['out2_paths'] = edges.assign(d=edges.txId2.map(out_deg).fillna(0)).groupby('txId1').d.sum().reindex(gf.index).fillna(0)

# longest path in / out and peel-chain runs (1 input, 1-2 outputs) via one pass in topological order
order = list(nx.topological_sort(G))
depth_up, chain = {}, {}
for v in order:
    preds = list(G.predecessors(v))
    depth_up[v] = max((depth_up[u] + 1 for u in preds), default=0)
    chain_like = len(preds) == 1 and 1 <= G.out_degree(v) <= 2
    chain[v] = chain[preds[0]] + 1 if chain_like else 0
depth_down = {}
for v in reversed(order):
    depth_down[v] = max((depth_down[w] + 1 for w in G.successors(v)), default=0)

gf['depth_up'] = gf.index.map(depth_up).fillna(0)
gf['depth_down'] = gf.index.map(depth_down).fillna(0)
gf['chain_run'] = gf.index.map(chain).fillna(0)
gf['n_anc'] = [len(nx.ancestors(G, n)) if n in G else 0 for n in gf.index]
gf['n_desc'] = [len(nx.descendants(G, n)) if n in G else 0 for n in gf.index]
struct_cols = list(gf.columns)
print(len(struct_cols), 'structural features')

12 structural features


## 3. Visibility-masked neighbour aggregates

In the test period, unlabelled transactions appear in the edge list but have **no feature rows**. Neighbour features therefore use only *visible* neighbours (labelled training rows and test rows), so a training transaction sees its neighbourhood exactly as a test transaction does. Neighbour **labels** are never used.

Mean and max of the 20 most important raw features (LightGBM gain, fitted on steps 1–20 only, so selection never sees a validation period) over funding (in) and spending (out) neighbours, plus visible-neighbour counts.

In [4]:
tr1 = lab[lab.time_step.isin(range(1, 21))]
imp_model = LGBMClassifier(random_state=SEED, verbose=-1, importance_type='gain').fit(tr1[feat_cols], tr1['label'])
top_feats = pd.Series(imp_model.feature_importances_, index=feat_cols).nlargest(20).index.tolist()

visible = pd.concat([lab[['txId'] + top_feats], test[['txId'] + top_feats]]).set_index('txId')
ve = edges[edges.txId1.isin(visible.index) & edges.txId2.isin(visible.index)]
in_agg = ve.join(visible, on='txId1').groupby('txId2')[top_feats].agg(['mean', 'max'])
in_agg.columns = [f'in_{f}_{s}' for f, s in in_agg.columns]
out_agg = ve.join(visible, on='txId2').groupby('txId1')[top_feats].agg(['mean', 'max'])
out_agg.columns = [f'out_{f}_{s}' for f, s in out_agg.columns]

nf = pd.DataFrame(index=gf.index)
nf['n_vis_in'] = ve.txId2.value_counts().reindex(nf.index).fillna(0)
nf['n_vis_out'] = ve.txId1.value_counts().reindex(nf.index).fillna(0)
nf = nf.join(in_agg).join(out_agg)           # NaN when no visible neighbour; LightGBM handles NaN natively
nbr_cols = list(nf.columns)

lab_x = lab.join(gf, on='txId').join(nf, on='txId')
test_x = test.join(gf, on='txId').join(nf, on='txId')
A_COLS = feat_cols + struct_cols + nbr_cols      # model A features (259)
print(len(nbr_cols), 'neighbour features |', len(A_COLS), 'model-A features')

82 neighbour features | 259 model-A features


## 4. Two-stage model

1. **Model A** (default LightGBM on the 259 features above) scores every visible transaction. Training rows get **out-of-fold** scores from 5 folds grouped by time step: edges never cross steps, so a transaction and all its neighbours are always scored by a model that never saw their labels. Prediction rows get scores from model A fitted on all training rows.
2. **Neighbour risk scores:** mean and max of model-A scores over in-, out- and all one-hop neighbours (6 features), and over two-hop funders-of-funders / spenders-of-spenders (4 features, used by exp11b only).
3. **Model B** = LightGBM with `extra_trees=True` (random split thresholds, a regulariser) on the 259 features + neighbour scores, probabilities averaged over 5 seeds.

| variant | model B settings | extra columns |
| --- | --- | --- |
| exp11a | 300 trees, learning rate 0.05, `min_child_samples` 50 | — |
| exp11b | as exp11a | two-hop risk scores |
| exp11c | 600 trees, learning rate 0.025, `min_child_samples` 50 | — |

In [5]:
DEFAULT_PARAMS = dict(random_state=SEED, verbose=-1)
SCORE_COLS = ['nb_in_mean', 'nb_in_max', 'nb_out_mean', 'nb_out_max', 'nb_all_mean', 'nb_all_max']
SCORE2_COLS = ['nb_in2_mean', 'nb_in2_max', 'nb_out2_mean', 'nb_out2_max']
STACK_COLS = A_COLS + SCORE_COLS
ET_BASE = dict(extra_trees=True, n_estimators=300, learning_rate=0.05, min_child_samples=50)
VARIANTS = {
    'exp11a': dict(b=ET_BASE, cols=STACK_COLS),
    'exp11b': dict(b=ET_BASE, cols=STACK_COLS + SCORE2_COLS),
    'exp11c': dict(b={**ET_BASE, 'n_estimators': 600, 'learning_rate': 0.025}, cols=STACK_COLS),
}

# all length-2 paths a -> b -> c (intermediate b may be any transaction)
paths2 = (edges.rename(columns={'txId1': 'a', 'txId2': 'b'})
               .merge(edges.rename(columns={'txId1': 'b', 'txId2': 'c'}), on='b')[['a', 'c']]
               .drop_duplicates())

def fit_predict(params, tr, va, cols):
    return LGBMClassifier(**params).fit(tr[cols], tr['label']).predict_proba(va[cols])[:, 1]

def seeded(extra, seed):
    return {**DEFAULT_PARAMS, **extra, 'random_state': seed}

def stage_a_scores(fit_rows, pred_rows, a_cols, n_splits=5):
    # out-of-fold scores for fit_rows (folds grouped by time step) + full-fit scores for pred_rows
    oof = np.zeros(len(fit_rows))
    for tr_i, va_i in GroupKFold(n_splits=n_splits).split(fit_rows, groups=fit_rows.time_step):
        m = LGBMClassifier(**DEFAULT_PARAMS).fit(fit_rows.iloc[tr_i][a_cols], fit_rows.iloc[tr_i]['label'])
        oof[va_i] = m.predict_proba(fit_rows.iloc[va_i][a_cols])[:, 1]
    m = LGBMClassifier(**DEFAULT_PARAMS).fit(fit_rows[a_cols], fit_rows['label'])
    full = m.predict_proba(pred_rows[a_cols])[:, 1]
    return pd.concat([pd.Series(oof, index=fit_rows.txId.values), pd.Series(full, index=pred_rows.txId.values)])

def neighbour_scores(score):
    # mean / max of model-A scores over in-, out- and all one-hop neighbours that have a score
    e = edges[edges.txId1.isin(score.index) & edges.txId2.isin(score.index)]
    into = pd.DataFrame({'node': e.txId2.values, 's': e.txId1.map(score).values})   # funders of node
    outof = pd.DataFrame({'node': e.txId1.values, 's': e.txId2.map(score).values})  # spenders of node
    out = pd.DataFrame(index=pd.Index(score.index, name='txId'))
    for name, frame in [('in', into), ('out', outof), ('all', pd.concat([into, outof]))]:
        agg = frame.groupby('node').s.agg(['mean', 'max']).reindex(out.index)
        out[f'nb_{name}_mean'], out[f'nb_{name}_max'] = agg['mean'].values, agg['max'].values
    return out

def two_hop_scores(score):
    # mean / max of model-A scores over visible two-hop funders (in2) and spenders (out2)
    p = paths2[paths2.a.isin(score.index) & paths2.c.isin(score.index)]
    out = pd.DataFrame(index=pd.Index(score.index, name='txId'))
    for name, node, other in [('in2', 'c', 'a'), ('out2', 'a', 'c')]:
        agg = p.assign(s=p[other].map(score)).groupby(node).s.agg(['mean', 'max']).reindex(out.index)
        out[f'nb_{name}_mean'], out[f'nb_{name}_max'] = agg['mean'].values, agg['max'].values
    return out

def score_features(fit_rows, pred_rows):
    sc = stage_a_scores(fit_rows, pred_rows, A_COLS)
    return neighbour_scores(sc).join(two_hop_scores(sc))

def rank_average(*preds):
    # equal-weight mean of normalised ranks (0-1]; ROC AUC depends only on ranking
    return np.mean([rankdata(p) / len(p) for p in preds], axis=0)

## 5. Validation: forward-in-time folds

Each fold trains on earlier steps and validates on later ones, mimicking the train (1–35) → test (36–49) gap. The whole two-stage procedure, including model A's out-of-fold scores, is rebuilt inside every fold, so validation labels never reach any feature.

Final models were chosen by the mean AUC over the recent folds F2–F4 (the folds that ordered the public scores correctly), rejecting any model whose F1 fell more than 0.02 below the stacked baseline (F1 ≥ 0.9312). Validation uses seeds 42–44, as in the experiment log.

In [6]:
FOLDS = {'F1': (range(1, 21), range(21, 28)), 'F2': (range(1, 28), range(28, 36)),
         'F3': (range(1, 25), range(25, 36)), 'F4': (range(1, 31), range(31, 36))}

if RUN_VALIDATION:
    t0 = time.time()
    cv_pred = {e: {} for e in VARIANTS}
    y = {}
    for f, (tr_steps, va_steps) in FOLDS.items():
        tr, va = lab_x[lab_x.time_step.isin(tr_steps)], lab_x[lab_x.time_step.isin(va_steps)]
        sf = score_features(tr, va)
        trj, vaj = tr.join(sf, on='txId'), va.join(sf, on='txId')
        y[f] = va['label'].values
        for e, v in VARIANTS.items():
            cv_pred[e][f] = np.mean([fit_predict(seeded(v['b'], s), trj, vaj, v['cols']) for s in SEEDS_CV], axis=0)
    cv_pred['exp11g'] = {f: rank_average(*[cv_pred[e][f] for e in VARIANTS]) for f in FOLDS}

    cv = pd.DataFrame({e: {f: roc_auc_score(y[f], p[f]) for f in FOLDS} for e, p in cv_pred.items()}).T
    cv['mean_F2_F4'] = cv[['F2', 'F3', 'F4']].mean(axis=1)
    cv['F1_guard_ok'] = cv['F1'] >= 0.9312
    cv['public_lb'] = cv.index.map({'exp11a': 0.96870, 'exp11b': 0.96772, 'exp11c': 0.96925, 'exp11g': 0.96873})
    print(f'validated in {time.time() - t0:.0f}s')
    display(cv.round(4))

validated in 726s


,F1,F2,F3,F4,mean_F2_F4,F1_guard_ok,public_lb
exp11a,0.9429,0.9983,0.9982,0.9989,0.9985,True,0.9687
exp11b,0.9367,0.9984,0.9984,0.9989,0.9986,True,0.9677
exp11c,0.9358,0.9984,0.9982,0.9989,0.9985,True,0.9692
exp11g,0.9385,0.9984,0.9983,0.9989,0.9985,True,0.9687


## 6. Final fit on all labelled steps 1–35 and submission files

Model A's out-of-fold scores cover all labelled training rows; model A fitted on all of them scores the test set. Each model-B variant is fitted with seeds 42–46 and its probabilities are averaged.

In [7]:
t0 = time.time()
sf_full = score_features(lab_x, test_x)
lab_d, test_d = lab_x.join(sf_full, on='txId'), test_x.join(sf_full, on='txId')

def write_submission(pred, name):
    sub = sample[['index']].copy()
    sub['target'] = pred
    assert sub.shape == sample.shape and list(sub.columns) == list(sample.columns)
    assert (sub['index'].values == sample['index'].values).all()
    assert np.isfinite(sub['target']).all() and sub['target'].between(0, 1).all()
    sub.to_csv(OUT / name, index=False)
    return OUT / name

files = {}
for e, v in VARIANTS.items():
    pred = np.mean([fit_predict(seeded(v['b'], s), lab_d, test_d, v['cols']) for s in SEEDS_FINAL], axis=0)
    files[e] = write_submission(pred, f'{e}_seedavg5.csv')

# exp11g: rank average of the three variants, computed from the written files
files['exp11g'] = write_submission(rank_average(*[pd.read_csv(files[e]).target.values for e in VARIANTS]),
                                   'exp11g_rankavg_11abc.csv')
print(f'fitted and written in {time.time() - t0:.0f}s')
for e in ['exp11c', 'exp11g']:
    print('FINAL SUBMISSION:', files[e])

fitted and written in 242s
FINAL SUBMISSION: submissions\final\exp11c_seedavg5.csv
FINAL SUBMISSION: submissions\final\exp11g_rankavg_11abc.csv


## 7. Reproducibility check

The two final files are compared with the versions submitted to Kaggle (kept in `submissions/`).

In [8]:
for e, name in [('exp11c', 'exp11c_seedavg5.csv'), ('exp11g', 'exp11g_rankavg_11abc.csv')]:
    submitted = Path('submissions') / name
    if submitted.exists():
        a, b = pd.read_csv(files[e]), pd.read_csv(submitted)
        diff = float(np.abs(a.target - b.target).max())
        print(f'{name}: max |difference| vs submitted file = {diff:.2e}', '(identical)' if diff < 1e-12 else '')
    else:
        print(f'{name}: submitted copy not found, skipping comparison')
print(f'total runtime {(time.time() - T0) / 60:.1f} min')

exp11c_seedavg5.csv: max |difference| vs submitted file = 0.00e+00 (identical)
exp11g_rankavg_11abc.csv: max |difference| vs submitted file = 0.00e+00 (identical)
total runtime 16.5 min
